# Analyse statistique

### Les facteurs associés à la performance des marathoniens

> **Mode d'emploi :** placez le fichier `dataSetMarathon.csv` dans le dossier `data_Set` situé à côté de ce notebook. Exécutez ensuite toutes les cellules dans l'ordre (`Kernel` → `Restart & Run All`). Les résultats chiffrés de ce document sont recalculés à partir du fichier CSV.

## <u>*Sommaire*</u>
1. Introduction + Question générale 
2. Préparation et qualité des données
   - Critique de la qualité des données
   - Nettoyage et création de variables
   - Présentation des variables et granularité
3. Analyse statistique descriptive
   - Distribution des temps de course
   - Temps selon le genre
   - Âge et performances
   - Comparaison par pays
4. Analyse inférentielle et tests d’hypothèses
   - Test de Welch : genre
   - Régression linéaire : âge
   - Analyse des résidus
   - Régression multiple : âge et genre
   - ANOVA de Welch : pays
   - Régression multiple : âge, genre et pays
5. Interprétation et conclusion

## <u>*1. INTRODUCTION :*</u>

Le marathon est une épreuve d’endurance exigeante qui demande aux coureurs de maintenir un effort important sur une distance de 42,195 kilomètres. 
La performance réalisée lors d’un marathon peut varier considérablement d’un coureur à l’autre. Ces différences de performance peuvent être liées à de nombreux facteurs, tels que l’âge, le sexe, l’expérience, le niveau d’entraînement, la vitesse moyenne ou encore certaines caractéristiques individuelles des coureurs

L’analyse permettra dans un premier temps de décrire les caractéristiques des coureurs et leur niveau de performance.
Elle cherchera ensuite à mettre en évidence les relations entre les différentes variables disponibles et le temps de course.
Des méthodes statistiques pourront notamment être utilisées afin de déterminer quels facteurs sont les plus fortement associés à la performance et d’évaluer leur importance.

### <u>La question que l’on se pose alors est la suivante :</u>

### ***Quels sont les facteurs qui influencent la performance d’un coureur lors d’un marathon, mesurée par son temps de course ?***

<br>

### Cadre d'interprétation

Dans ce notebook, le temps de course est la variable expliquée. L'âge, le genre et le pays sont étudiés comme variables associées au temps. Le terme « associé » est privilégié à « influence » : les analyses observationnelles ne permettent pas, à elles seules, d'établir une causalité.

Les tests utilisent un seuil de significativité fixé à $\alpha = 0,05$. Une p-valeur inférieure à ce seuil conduit au rejet de l'hypothèse nulle ; elle ne mesure pas l'importance pratique de l'écart.

## <u>2. Préparation et qualité des données</u>

##### <u>*Critique de la qualité des données :*</u>

Dans l’ensemble, le jeu de données est relativement complet. La majorité des variables ne contiennent aucune donnée aberrante. Cependant, certaines valeurs sont absentes.

Les performances présentes dans le fichier sont majoritairement comprises entre environ 2 h et 2 h 30. Le jeu de données semble donc surtout décrire des performances de haut niveau ; cette observation ne permet toutefois pas de caractériser tous les marathoniens. 

Une autre remarque concerne la variable Event : elle ne contient qu’une seule modalité, « Marathon ». Cette variable n'apporte donc aucune information pour différencier les observations et ne sera pas utile dans l’analyse statistique.

La variable Name comporte également de nombreux noms différents, mais elle n’est pas directement pertinente pour expliquer la performance. Elle pourra principalement servir à identifier les observations.

Enfin, certaines variables devront être transformées avant l’analyse. Par exemple, la variable Time est actuellement enregistrée sous la forme « heures:minutes ». Pour réaliser des calculs statistiques, il sera préférable de la convertir en une durée numérique, par exemple en secondes ou en minutes. De même, les variables Date et Date of Birth devront être converties dans un format de date afin de pouvoir calculer notamment l’âge des coureurs au moment de la course.

##### <u>*Nettoyage et création de variables :*</u>

1) L’âge du marathonien au moment de sa participation au marathon  
2) Sa vitesse moyenne sur le marathon.

Ces deux nouvelles variables vont nous permettre d’être plus précis dans la réalisation de nos statistiques descriptives, en apportant davantage d’informations sur les caractéristiques des participants et leurs performances.

Le nombre d'observations avant et après le nettoyage a été comparé afin de quantifier les suppressions effectuées. 


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

# Chargement du fichier : les 10 premières colonnes utiles sont conservées.
dataFrame = pd.read_csv(
    "data_Set/dataSetMarathon.csv",
    delimiter=";",
    usecols=range(10)
)

lignes_avant = len(dataFrame)
print("Nombre de lignes avant nettoyage :", lignes_avant)

dataFrame.columns = dataFrame.columns.str.strip()

# Retirer les lignes sans date de naissance ou sans date/temps exploitables.
dataFrame = dataFrame.dropna(subset=["Date of Birth", "Date", "Time"]).copy()
dataFrame["Date of Birth"] = pd.to_datetime(
    dataFrame["Date of Birth"], dayfirst=True, errors="coerce"
)
dataFrame["Date"] = pd.to_datetime(
    dataFrame["Date"], dayfirst=True, errors="coerce"
)
dataFrame["Time"] = pd.to_timedelta(dataFrame["Time"], errors="coerce")

# Écarter les lignes dont une conversion a échoué.
dataFrame = dataFrame.dropna(subset=["Date of Birth", "Date", "Time"]).copy()

# Âge révolu au jour de la course.
anniversaire_non_atteint = (
    (dataFrame["Date"].dt.month < dataFrame["Date of Birth"].dt.month)
    | (
        (dataFrame["Date"].dt.month == dataFrame["Date of Birth"].dt.month)
        & (dataFrame["Date"].dt.day < dataFrame["Date of Birth"].dt.day)
    )
)
dataFrame["Age"] = (
    dataFrame["Date"].dt.year
    - dataFrame["Date of Birth"].dt.year
    - anniversaire_non_atteint.astype(int)
)

# Temps conservé sous forme de durée ; Temps_h facilite les calculs statistiques.
dataFrame["Temps_h"] = dataFrame["Time"].dt.total_seconds() / 3600
distance_marathon = 42.195
dataFrame["Speed"] = distance_marathon / dataFrame["Temps_h"]

# Écarter les âges ou durées manifestement impossibles/non exploitables.
dataFrame = dataFrame[
    dataFrame["Age"].between(0, 100)
    & dataFrame["Temps_h"].gt(0)
    & np.isfinite(dataFrame["Speed"])
].copy()

lignes_apres = len(dataFrame)
ligne_supprimee = lignes_avant - lignes_apres

print("Nombre de lignes après nettoyage :", lignes_apres)
print("Nombre de lignes supprimées :", ligne_supprimee)
print(f"Pourcentage conservé : {100 * lignes_apres / lignes_avant:.2f} %")

# Variable utilisée dans les analyses : temps en heures.
temps = dataFrame["Temps_h"]

display(dataFrame.head(20))

##### Présentation des variables et granularité

Une ligne représente une performance réalisée par un coureur lors d'une course. Le même athlète peut apparaître plusieurs fois : les observations ne sont donc pas nécessairement indépendantes.

- `Rank` : rang fourni.
- `Time` : temps de course d'origine.
- `Name` : nom utilisé pour identifier les observations.
- `Country` : code pays (variable qualitative).
- `Date of Birth` : date de naissance.
- `Place` : position finale dans l'épreuve.
- `City` : ville de la compétition (variable qualitative).
- `Date` : date de l'épreuve.
- `Gender` : catégorie hommes/femmes telle que fournie dans le fichier (variable qualitative).
- `Event` : type d'épreuve.
- `Age` : âge en années révolues au moment de la course (variable quantitative).
- `Temps_h` : temps de course converti en heures (variable quantitative).
- `Speed` : vitesse moyenne en km/h (variable quantitative, dérivée du temps).

La vitesse est mathématiquement liée au temps et ne constitue pas un facteur explicatif indépendant du temps. Le jeu de données est orienté vers des performances de haut niveau, avec une représentation inégale des pays, des genres et des années.

Le diagramme circulaire ci-dessous présente la proportion d'observations conservées et supprimées pendant le nettoyage. Les effectifs et pourcentages sont calculés automatiquement à partir du fichier chargé.

In [ ]:
valeurs = [lignes_apres, ligne_suprimee]
labels = ["Observations conservées", "Observations supprimées"]
couleurs = ["#4CAF50", "#E57373"]

fig, ax = plt.subplots(figsize=(6, 4.5))

parts, textes, pourcentages = ax.pie(
    valeurs,
    labels=None,
    colors=couleurs,
    autopct="%1.2f%%",
    startangle=90,
    pctdistance=0.65,
    textprops={"fontsize": 9}
)

ax.legend(
    parts,
    [
        f"Conservées : {lignes_apres:,}".replace(",", " "),
        f"Supprimées : {ligne_suprimee:,}".replace(",", " ")
    ],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    fontsize=9,
    frameon=False
)

fig.suptitle(
    "Répartition des observations après nettoyage",
    fontsize=11,
    x=0.5,
    y=0.95,
    ha="center"
)

ax.axis("equal")
plt.tight_layout()
plt.show()


## <u>3.  Analyse descriptive </u> :

Nous allons d'abord décrire les temps, puis les comparer selon le genre et examiner les associations avec l'âge. Nous étudierons ensuite les pays suffisamment représentés. Chaque graphique sera accompagné d'une interprétation. Les analyses portent sur les performances enregistrées dans le fichier, qui ne constituent pas nécessairement un échantillon représentatif de tous les marathoniens.

#### <u>*Distribution des temps de course*</u> ####

Nous utilisons la moyenne et la médiane pour décrire le centre de la distribution, l'écart-type et les quartiles pour la dispersion, puis un histogramme et une boîte à moustaches pour identifier l'asymétrie et les valeurs extrêmes. Les valeurs extrêmes signalées par le boxplot ne sont pas automatiquement des erreurs

In [ ]:
temps = dataFrame['Temps_h']

resume_temps = pd.Series({
    'Effectif': temps.count(),
    'Moyenne (h)': temps.mean(),
    'Médiane (h)': temps.median(),
    'Écart-type (h)': temps.std(),
    'Minimum (h)': temps.min(),
    'Premier quartile (h)': temps.quantile(0.25),
    'Troisième quartile (h)': temps.quantile(0.75),
    'Maximum (h)': temps.max(),
})
print(resume_temps.round(3).to_string())

q1 = temps.quantile(0.25)
q3 = temps.quantile(0.75)
ecart_interquartile = q3 - q1
limite_basse = q1 - 1.5 * ecart_interquartile
limite_haute = q3 + 1.5 * ecart_interquartile
nombre_atypiques = ((temps < limite_basse) | (temps > limite_haute)).sum()
print(f'Valeurs atypiques (règle 1,5 × IQR) : {nombre_atypiques}')

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(temps.dropna(), bins=35, edgecolor='white')
ax.axvline(temps.mean(), color='red', linestyle='--', label='Moyenne')
ax.axvline(temps.median(), color='black', linestyle=':', label='Médiane')
ax.set(xlabel='Temps (heures)', ylabel='Nombre de performances',
       title='Distribution des temps de marathon')
ax.legend()
plt.tight_layout()
plt.show()



Le graphe représente 16 819 temps de marathon (moyenne 2,258 h, médiane 2,186 h). L'histogramme est bimodal : un pic net vers 2,18-2,20 h et un second groupe étalé de 2,3 à 2,5 h.

Les statistiques seules laissent croire que les données sont homogènes. L'histogramme révèle en réalité deux groupes de coureurs, et que la moyenne ne correspond à aucun d'eux

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3))

ax.boxplot(
    temps.dropna(),
    orientation='horizontal'
)

ax.set(
    xlabel='Temps (heures)',
    title='Dispersion des temps de course'
)

plt.tight_layout()
plt.show()

Ce boxplot représente la dispersion des temps de course, de 1,99 h à 2,5 h. La boîte va de Q1 (2,156 h) à Q3 (2,418 h), avec la médiane (2,186 h) tout à gauche dans la boîte, et aucun point au-delà des moustaches : il n'y a pas de valeur aberrante.

Le boxplot ajoute une vue claire des quartiles, de la médiane et des valeurs aberrantes, utile pour comparer des groupes, mais il ne révèle pas la bimodalité que l'histogramme montrait.

#### <u>*Temps de course selon le genre*</u> ####

On compare l'effectif, la moyenne, la médiane et l'écart-type ainsi que les distributions. Une différence descriptive observée n'est pas encore une conclusion inférentielle. Le test sera effectué plus loin.

In [ ]:

fig, ax = plt.subplots(figsize=(8, 4))
fbins = np.linspace(temps.min(), temps.max(), 31)

for genre, groupe in dataFrame.groupby('Gender'):
    temps_groupe = temps.loc[groupe.index].dropna()
    ax.hist(
        temps_groupe,
        bins=fbins,
        alpha=0.45,
        label=f"{genre} (n={len(temps_groupe):,})".replace(",", " ")
    )
ax.set(
    xlabel='Temps (heures)',
    ylabel='Nombre de marathoniens',
    title='Distribution des temps par genre'
)

ax.legend()
plt.tight_layout()
plt.show()

Ce graphe sépare les 16 819 temps de marathon selon le genre : 10 908 hommes et 5 911 femmes. Les hommes se concentrent entre 2,0 et 2,2 h, avec un pic vers 2,2 h, tandis que les femmes sont entre 2,3 et 2,5 h, avec un effectif qui augmente jusqu'à la coupure à 2,5 h. Il y a un vide entre les deux groupes, presque sans chevauchement.

Il explique la bimodalité vue sur le premier histogramme : les deux pics correspondent aux hommes et aux femmes, ce qui n'était qu'une hypothèse avant. Il montre aussi qu'analyser tous les coureurs ensemble est trompeur, puisque la moyenne globale ne représente aucun des deux groupes. Il est donc plus pertinent de calculer les statistiques séparément pour chaque genre.

In [ ]:
# Récupération des temps selon le genre
temps_hommes = temps[dataFrame['Gender'] == 'Men'].dropna()
temps_femmes = temps[dataFrame['Gender'] == 'Women'].dropna()

# Création du graphique
fig, ax = plt.subplots(figsize=(10, 4))

ax.boxplot(
    [temps_hommes, temps_femmes],
    orientation='horizontal',
    tick_labels=['Hommes', 'Femmes']
)

# Personnalisation du graphique
ax.set(
    xlabel='Temps de course (heures)',
    ylabel='Genre',
    title='Comparaison des temps de marathon selon le genre'
)

plt.tight_layout()
plt.show()

Ce boxplot compare les temps de marathon des femmes et des hommes.Les hommes ont une boîte entre 2,14 et 2,19 h (médiane à 2,17 h), les femmes entre 2,41 et 2,48 h (médiane à 2,45 h).
 Chaque groupe a de nombreuses valeurs aberrantes du côté des temps rapides.

Il permet de comparer directement les deux genres sur la même échelle : la médiane, la dispersion et l'absence de chevauchement des boîtes se voient immédiatement. Il ajoute aussi que, une fois les groupes séparés, des valeurs aberrantes apparaissent alors que le boxplot global n'en montrait aucune. En revanche, il ne montre pas la forme de la distribution comme l'histogramme précédent.

#### <u>*Age et performances*</u> ####

Un nuage de points permet de rechercher une tendance entre l'âge et la vitesse moyenne. On calcule aussi la corrélation de Pearson entre l'âge et le temps et on compare les médianes par tranches d'âge. Les coureurs et les compétitions ne sont pas tirés au hasard.

In [ ]:
analyse_age = pd.DataFrame({
    "Age": dataFrame["Age"],
    "Temps": temps,
    "Vitesse": dataFrame["Speed"],
}).dropna()

# Pearson mesure l'association linéaire ; Spearman mesure l'association monotone.
pearson = stats.pearsonr(analyse_age["Age"], analyse_age["Temps"])
spearman = stats.spearmanr(analyse_age["Age"], analyse_age["Temps"])

print(f"Pearson : r = {pearson.statistic:.4f} ; p-valeur = {pearson.pvalue:.3g}")
print(f"Spearman : rho = {spearman.statistic:.4f} ; p-valeur = {spearman.pvalue:.3g}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(analyse_age["Age"], analyse_age["Temps"], alpha=0.12, s=8)
ax.set(
    xlabel="Âge (années)",
    ylabel="Temps de course (heures)",
    title="Relation entre l'âge et le temps de course"
)
plt.tight_layout()
plt.show()

Ce nuage de points relie l'âge des coureurs (14 à 50 ans) à leur vitesse moyenne (17 à 21 km/h). Les points forment deux bandes horizontales, une vers 19-20,5 km/h et une vers 17-18,5 km/h. La plupart des coureurs ont entre 20 et 35 ans, et la vitesse baisse légèrement après 35-40 ans.

Spearman classe les valeurs par rang et vérifie si l'âge et le temps évoluent dans le même sens. C'est adapté à notre graphe car il ne demande pas une relation en ligne droite et résiste aux valeurs extrêmes.

In [ ]:
bornes = [0, 25, 35, 45, 55, 65, 120]
etiquettes = ['< 25', '25–34', '35–44', '45–54', '55–64', '65+']
tranches_age = pd.cut(dataFrame['Age'], bins=bornes, labels=etiquettes, right=False)

resume_age = temps.groupby(tranches_age, observed=True).agg(
    effectif='count', moyenne='mean', mediane='median'
)
print(resume_age.round(3).to_string())

print("L'analyse des temps de course par tranche d'âge montre que les performances médianes restent relativement proches pour les marathoniens de moins de 45 ans. La tranche des 25–34 ans présente le temps médian le plus faible, avec 2,185 heures.\n"
"À partir de 45 ans, on observe une augmentation plus importante du temps médian. Cependant, cette tranche ne comporte que 25 observations, ce qui limite la fiabilité de la comparaison.\n Ces résultats suggèrent que l'âge pourrait être associé aux performances, mais les différences observées restent faibles dans les tranches les plus représentées.")

#### <u>*Comparaison descriptive par pays*</u>

Afin d'éviter les comparaisons basées sur quelques individus, nous ne présentons que les pays comptant au moins 100 performances. Ce seuil est un choix descriptif explicite et non une garantie de représentativité. Les différences peuvent refléter la composition du fichier et la sélection des performances.

In [ ]:
effectifs_pays = dataFrame['Country'].value_counts()
pays_retenus = effectifs_pays[effectifs_pays >= 100].index
selection = dataFrame['Country'].isin(pays_retenus)

resume_pays = temps[selection].groupby(dataFrame.loc[selection, 'Country']).agg(
    effectif='count', mediane='median'
).sort_values('mediane')
print(resume_pays.round(3).to_string())

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(resume_pays.index, resume_pays['mediane'])
ax.set(xlabel='Pays (code)', ylabel='Temps médian (heures)',
       title='Temps médian par pays (au moins 100 performances)')
plt.xticks(rotation=45)
# L'axe commence à zéro pour ne pas exagérer visuellement les écarts.
ax.set_ylim(0, resume_pays['mediane'].max() * 1.10)
plt.tight_layout()
plt.show()

resume_pays_genre = temps[selection].groupby([
    dataFrame.loc[selection, 'Country'],
    dataFrame.loc[selection, 'Gender'],
]).agg(effectif='count', mediane='median')

print('Pays / genre avec au moins 30 performances :')
print(resume_pays_genre[resume_pays_genre['effectif'] >= 30].round(3).to_string())

## <u>4. Application des outils statistiques (inférence)</u> 

**Problématique :** *Quels sont les facteurs associés à la performance d’un coureur lors d’un marathon, mesurée par son temps de course ?*

Nous testons d'abord une différence de **temps moyen selon le genre**, puis une association **linéaire entre l'âge et le temps**. Les résultats décrivent des associations et ne prouvent pas un lien de cause à effet.

**Unité d'analyse du test principal :** comme plusieurs courses peuvent appartenir au même athlète, nous gardons son **meilleur temps enregistré** (identifié ici par nom, date de naissance et genre). Ce choix réduit le problème de répétition, mais l'identification peut rester imparfaite et les performances sélectionnées ne sont pas représentatives de tous les marathoniens.

#### <u>*Test d’hypothèse : les temps moyens diffèrent-ils selon le genre ?*</u> ####

**Variables :** `Gender` (qualitative, groupes `Men` et `Women`) et `Time` (quantitative, convertie temporairement en heures).

**Hypothèses :**
- **H₀ :** les temps moyens des hommes et des femmes sont égaux, soit μH = μF.
- **H₁ :** les temps moyens sont différents, soit μH ≠ μF.

**Choix d'un test bilatéral :** nous cherchons une différence dans un sens **ou** dans l'autre, sans imposer à l'avance quel groupe devrait être le plus rapide.

**Seuil :** α = 0,05 (risque de 5 % de rejeter H₀ à tort lorsqu'elle est vraie).

**Test retenu :** test t de **Welch**, qui compare deux moyennes sans supposer des variances identiques. Il convient mieux qu'un test t à variances communes en présence d'effectifs et de dispersions différents. Il suppose néanmoins des observations indépendantes et une situation dans laquelle l'approximation des moyennes est applicable.

**Statistique de test :**

\[ t = \frac{\overline{x}_H-\overline{x}_F}{\sqrt{s_H^2/n_H+s_F^2/n_F}} \]

Les degrés de liberté sont estimés par la formule de Welch–Satterthwaite.

**Règle de décision :** nous rejetons H₀ si `p < 0,05` **et**, de façon équivalente, si la statistique t se situe en dehors de l'intervalle `[-t_critique ; +t_critique]` (région de rejet bilatérale).

In [ ]:
# 1. Garder le meilleur temps enregistré pour chaque athlète.
# Cela évite de compter ses différentes courses comme des individus distincts.
donnees_test = dataFrame[['Name', 'Date of Birth', 'Gender']].copy()
donnees_test['Temps'] = temps

donnees_test = donnees_test.dropna()
donnees_test = donnees_test.groupby(
    ['Name', 'Date of Birth', 'Gender'],
    as_index=False
)['Temps'].min()

# 2. Séparer les deux groupes.
hommes = donnees_test.loc[donnees_test['Gender'] == 'Men', 'Temps']
femmes = donnees_test.loc[donnees_test['Gender'] == 'Women', 'Temps']

# 3. Calculer les effectifs, moyennes et variances.
n_hommes = len(hommes)
n_femmes = len(femmes)
moy_hommes = hommes.mean()
moy_femmes = femmes.mean()
var_hommes = hommes.var(ddof=1)
var_femmes = femmes.var(ddof=1)

# 4. Calculer la statistique t de Welch.
part_hommes = var_hommes / n_hommes
part_femmes = var_femmes / n_femmes

erreur_standard = np.sqrt(part_hommes + part_femmes)
difference = moy_hommes - moy_femmes
t_observe = difference / erreur_standard

# 5. Calculer les degrés de liberté et la p-valeur bilatérale.
degrés_liberte = (part_hommes + part_femmes) ** 2 / (
    part_hommes ** 2 / (n_hommes - 1)
    + part_femmes ** 2 / (n_femmes - 1)
)

alpha = 0.05
p_valeur = 2 * stats.t.sf(abs(t_observe), df=degrés_liberte)
t_critique = stats.t.ppf(1 - alpha / 2, df=degrés_liberte)

# 6. Intervalle de confiance à 95 % de la différence de moyennes.
marge = t_critique * erreur_standard
ic_min = (difference - marge) * 60
ic_max = (difference + marge) * 60

print(f'Effectifs : {n_hommes} hommes et {n_femmes} femmes')
print(f'Temps moyen hommes : {moy_hommes:.4f} heure(s)')
print(f'Temps moyen femmes : {moy_femmes:.4f} heure(s)')
print(f'Différence hommes - femmes : {difference * 60:.2f} minutes')
print(f'Statistique t observée : {t_observe:.3f}')
print(f'Degrés de liberté approchés : {degrés_liberte:.1f}')
print(f'P-valeur : {p_valeur:.3g} (une valeur affichée à 0 correspond à une limite numérique)')
print(f'Valeurs critiques : {-t_critique:.3f} et {t_critique:.3f}')
print(f'IC à 95 % de la différence : [{ic_min:.2f} ; {ic_max:.2f}] minutes')

# 7. Décision par les deux méthodes exigées par la grille.
print('Décision selon la p-valeur :',
      'rejet de H0' if p_valeur < alpha else 'non-rejet de H0')
print('Décision selon la région de rejet :',
      'rejet de H0' if abs(t_observe) > t_critique else 'non-rejet de H0')


#### <u>*Interprétation du test de Welch*</u> ####

**Résultats calculés sur le CSV fourni (une meilleure performance par athlète) :**

- 2 966 athlètes dans le groupe hommes et 1 511 dans le groupe femmes.
- Moyennes : environ **2,153 h** pour les hommes et **2,428 h** pour les femmes, soit une différence de **16,50 minutes** (hommes – femmes = −16,50 min).
- Statistique observée : **t ≈ −175,57** ; valeur critique : **±1,961**.
- P-valeur : **extrêmement petite** (affichée comme `0` par le logiciel à cause de sa précision numérique), donc inférieure à 0,05.

**Méthode 1, p-valeur :** puisque `p < 0,05`, nous **rejetons H₀**.

**Méthode 2, région de rejet :** puisque `−175,57 < −1,961`, la statistique est dans la région de rejet. Nous **rejetons également H₀**.

**Conclusion dans le contexte :** dans les meilleures performances enregistrées par athlète dans ce dataset, les temps moyens diffèrent statistiquement entre les deux groupes. Les hommes ont en moyenne un temps inférieur d'environ **16,5 minutes**. Ce résultat ne permet pas de généraliser à tous les marathoniens ni de conclure que le genre est une cause isolée : les données sont sélectionnées et d'autres facteurs (niveau sportif, âge, année, parcours...) peuvent intervenir.

**Choix de gouvernance :** si un organisateur utilise ce type de données pour communiquer des statistiques de performance ou construire des références, il devrait **présenter des résultats par catégorie pertinente** plutôt qu'une moyenne unique, et vérifier les critères de sélection avant de généraliser.

#### <u>*Régression linéaire : l’âge est-il associé au temps ?*</u> ####

Nous complétons le test principal par une **régression linéaire simple** :

\[\text{Temps estimé} = b_0 + b_1 \times \text{Âge}\]

- **H₀ :** la pente de la droite est nulle (`β₁ = 0`), donc pas d'association *linéaire* dans la population.
- **H₁ :** la pente est non nulle (`β₁ ≠ 0`). Le test est **bilatéral**, car l'association pourrait être positive ou négative.
- **Seuil :** α = 0,05.

Le coefficient **$R^2$** décrit la proportion de variation des temps que peut expliquer cette droite. Le test de la pente évalue la significativité statistique ; une faible p-valeur ne signifie pas nécessairement une forte relation.

**Remarque :** ce modèle est illustratif et repose sur les performances enregistrées, avec parfois plusieurs observations par personne. Les p-valeurs naïves peuvent surestimer la précision de l'inférence lorsque les performances sont répétées.

In [ ]:
# Ajuster la régression linéaire âge -> temps.
regression = stats.linregress(analyse_age['Age'], analyse_age['Temps'])

pente = regression.slope
r_carre = regression.rvalue ** 2
t_pente = pente / regression.stderr

# Test bilatéral de la pente.
n = len(analyse_age)
alpha = 0.05
t_critique_pente = stats.t.ppf(1 - alpha / 2, df=n - 2)

print(f'Pente : {pente:.6f} heure par année')
print(f'Pente : {pente * 60:.3f} minute par année')
print(f'Coefficient R² : {r_carre:.5f}')
print(f'Statistique t : {t_pente:.3f}')
print(f'P-valeur : {regression.pvalue:.5g}')
print(f'Région de rejet : t < {-t_critique_pente:.3f} ou t > {t_critique_pente:.3f}')

print('Décision p-valeur :',
      'rejet de H0' if regression.pvalue < alpha else 'non-rejet de H0')
print('Décision région de rejet :',
      'rejet de H0' if abs(t_pente) > t_critique_pente else 'non-rejet de H0')

# Représenter les performances et la droite ajustée.
ages = np.linspace(analyse_age['Age'].min(), analyse_age['Age'].max(), 100)
temps_estimes = regression.intercept + pente * ages

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(analyse_age['Age'], analyse_age['Temps'], s=8, alpha=0.1)
ax.plot(ages, temps_estimes, color='red', label='Régression linéaire')
ax.set(
    xlabel='Âge (années)',
    ylabel='Temps de course (heures)',
    title='Régression linéaire : âge et temps'
)
ax.legend()
plt.tight_layout()
plt.show()


#### <u>*Interprétation de la régression*</u> ####

La corrélation descriptive âge–temps était d'environ **r = 0,023**, soit **$R^2$ ≈ 0,0005** (environ **0,05 %** de la variation expliquée par l'âge seul dans le modèle linéaire). Même si la pente peut être statistiquement différente de zéro, elle a **très peu d'intérêt prédictif** dans ce dataset. L'âge seul ne suffit donc pas à prévoir convenablement un temps de marathon.

La relation peut ne pas être linéaire : comparer aussi les tranches d'âge reste utile. Pour interpréter une éventuelle p-valeur faible, il faut tenir compte des mesures répétées et de la sélection des performances.

#### <u>*Contrôle du modèle : analyse des résidus$*</u> ####

Un **résidu** est l'écart entre le temps observé et le temps estimé par la droite. Nous examinons leur répartition pour voir si un modèle linéaire est raisonnable. Un tracé structuré, une dispersion variable ou des valeurs très atypiques suggèrent que le modèle n'est pas suffisant.

In [ ]:
# Calculer les résidus : temps réel moins temps prédit.
temps_predits = regression.intercept + pente * analyse_age['Age']
residus = analyse_age['Temps'] - temps_predits

# Résidus en fonction des valeurs prévues.
fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(temps_predits, residus, s=8, alpha=0.12)
ax.axhline(0, linestyle='--', color='red')
ax.set(
    xlabel='Temps prédit (heures)',
    ylabel='Résidu (heures)',
    title='Répartition des résidus'
)
plt.tight_layout()
plt.show()

# Distribution des résidus.
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(residus, bins=40, edgecolor='white')
ax.set(
    xlabel='Résidu (heures)',
    ylabel='Nombre de performances',
    title='Distribution des résidus'
)
plt.tight_layout()
plt.show()


#### <u>*Régression linéaire multiple : rôle de l’âge et du genre*</u> ####

Notre problématique porte sur **plusieurs facteurs** associés au temps de marathon. La régression précédente étudiait seulement l’âge. Nous complétons donc l’analyse par une **régression linéaire multiple**, méthode présentée dans le cours *Séance de synthèse : Statistiques* (pages 23 à 27).

Nous expliquons le **temps en heures** par l'**âge en années** et le **genre** (catégories `Men` et `Women`). Le modèle s'écrit :

\[
\widehat{Temps} = b_0 + b_1\times Age + b_2\times I(Gender=Women)
\]

La variable indicatrice vaut **0 pour Men** (catégorie de référence) et **1 pour Women**. Ainsi :

- **b₁** : variation de temps estimée pour une année d'âge supplémentaire, **à genre identique** ;
- **b₂** : différence de temps estimée pour Women par rapport à Men, **à âge identique** ;
- **$R^2$** : proportion de la variabilité du temps expliquée ensemble par les deux variables.

Pour éviter de considérer plusieurs résultats d'une même personne comme indépendants, nous retenons **une seule observation par athlète identifiable : sa meilleure performance enregistrée** (nom, date de naissance et genre). Il s'agit donc d'une analyse des *meilleures performances disponibles*, pas de l'ensemble des résultats. Cette sélection peut introduire un biais et ne démontre aucune causalité.

**Hypothèses des tests de coefficients :** H₀ : β₁ = 0 contre H₁ : β₁ ≠ 0 pour l'âge, et H₀ : β₂ = 0 contre H₁ : β₂ ≠ 0 pour le genre. Les tests sont bilatéraux, au seuil α = 0,05. Nous utilisons des erreurs-types robustes (HC3) pour ne pas supposer une variance des résidus identique partout. Les limites de représentativité et les variables non prises en compte subsistent.

In [ ]:

donnees_modele = dataFrame[[
    'Name', 'Date of Birth', 'Gender', 'Age'
]].copy()
donnees_modele['Temps'] = temps

donnees_modele = donnees_modele.dropna()
donnees_modele = donnees_modele[
    donnees_modele['Gender'].isin(['Men', 'Women'])
]

donnees_modele = (
    donnees_modele.sort_values('Temps')
    .drop_duplicates(subset=['Name', 'Date of Birth', 'Gender'])
    .copy()
)

# Ajuster le modèle, avec Men comme catégorie de référence.
modele_multiple = smf.ols(
    'Temps ~ Age + C(Gender, Treatment(reference="Men"))',
    data=donnees_modele
).fit(cov_type='HC3')

# Récupérer les coefficients sans recopier le nom interne de la variable catégorielle.
nom_genre = next(
    nom for nom in modele_multiple.params.index if 'Gender' in nom
)

coefficient_age = modele_multiple.params['Age']
coefficient_genre = modele_multiple.params[nom_genre]

print(f'Nombre d’athlètes : {len(donnees_modele)}')
print(f'Coefficient âge : {coefficient_age * 60:.3f} min par année')
print(f'Écart Women - Men à âge identique : {coefficient_genre * 60:.2f} min')
print(f'R² : {modele_multiple.rsquared:.4f}')
print(f'R² ajusté : {modele_multiple.rsquared_adj:.4f}')

# Présenter les coefficients, les p-valeurs et leurs intervalles de confiance.
intervalles = modele_multiple.conf_int()
resultats_modele = pd.DataFrame({
    'Coefficient (min)': modele_multiple.params * 60,
    'IC 95 % - bas (min)': intervalles[0] * 60,
    'IC 95 % - haut (min)': intervalles[1] * 60,
    'P-valeur': modele_multiple.pvalues
})
print('\nRésultats de la régression multiple :')
print(resultats_modele.round(4).to_string())

# Comparer le modèle complet aux modèles avec un seul facteur.
modele_age = smf.ols('Temps ~ Age', data=donnees_modele).fit()
modele_genre = smf.ols('Temps ~ C(Gender)', data=donnees_modele).fit()

print('\nComparaison des R² sur les mêmes athlètes :')
print(f'Âge seul : {modele_age.rsquared:.4f}')
print(f'Genre seul : {modele_genre.rsquared:.4f}')
print(f'Âge et genre : {modele_multiple.rsquared:.4f}')


#### <u>*Visualisation des deux relations estimées*</u> ####

Nous représentons les **temps prédits** par le modèle pour les deux catégories, en utilisant les mêmes âges sur l'axe horizontal. L'écart vertical entre les deux droites traduit la différence associée au genre *à âge identique*. La pente commune traduit l'association linéaire de l'âge *à genre identique*. Le modèle suppose ici des droites parallèles : il ne teste pas si la pente diffère selon le genre.

In [ ]:
# Prédire les temps selon l'âge pour les deux catégories.
ages_graphe = np.linspace(
    donnees_modele['Age'].min(),
    donnees_modele['Age'].max(),
    100
)

fig, ax = plt.subplots(figsize=(9, 4))

for genre in ['Men', 'Women']:
    valeurs = pd.DataFrame({
        'Age': ages_graphe,
        'Gender': genre
    })
    predictions = modele_multiple.predict(valeurs)
    ax.plot(ages_graphe, predictions, label=genre, linewidth=2)

ax.set(
    xlabel='Âge (années)',
    ylabel='Temps de course prédit (heures)',
    title='Régression multiple : temps estimé selon âge et genre'
)
ax.legend(title='Genre')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()


#### <u>*Interprétation de la régression multiple (sur 4 477 athlètes identifiables)*</u> ####

- **Genre à âge identique :** la catégorie `Women` est associée à un temps estimé **16,49 minutes plus long** que la catégorie `Men`, avec un intervalle de confiance à 95 % d’environ **[16,31 ; 16,68] minutes**. La p-valeur est extrêmement petite. Cela décrit une association dans les données et **ne démontre pas une cause**.
- **Âge à genre identique :** le coefficient estimé est **+0,016 minute par an** (environ une seconde par an), avec un intervalle de confiance à 95 % allant d'environ **−0,002 à +0,033 minute par an**. La p-valeur vaut **0,0755** : au seuil de 5 %, **nous ne rejetons pas H₀** pour la pente d'âge dans ce modèle.
- **Qualité d'ajustement :** le modèle complet donne **$R^2$ = 0,9008** ; sur les **mêmes athlètes**, le modèle avec le genre seul donne **$R^2$ = 0,9007** et l'âge seul **$R^2$ = 0,0017**. L'ajout de l'âge améliore donc à peine l'ajustement. La valeur très élevée de $R^2$ reflète surtout la séparation des catégories **dans ce fichier de performances d'élite sélectionnées**, et ne permet pas de prédire les performances de tous les marathoniens.

La sélection d'une meilleure performance par athlète peut elle-même favoriser les personnes avec davantage de résultats dans le fichier. De plus, les différences liées au parcours, à l'année, à l'expérience et à l'entraînement ne sont pas contrôlées. Nous ne devons donc pas généraliser les coefficients à l'ensemble des coureurs.

#### <u>*Vérification des résidus de la régression multiple*</u> ####

Le cours présente quatre conditions du modèle linéaire : **linéarité, indépendance des erreurs, variance constante des erreurs et normalité des erreurs** (séance de synthèse, page 26). Les graphiques suivants permettent de rechercher des écarts importants aux conditions de linéarité et de dispersion constante, ainsi que des asymétries. Ils ne prouvent pas à eux seuls que toutes les hypothèses sont satisfaites. Les erreurs-types HC3 atténuent les problèmes liés à une variance non constante, mais pas ceux liés à la sélection des performances.

In [ ]:
# Comparer les résidus aux temps prédits.
predictions_multiple = modele_multiple.fittedvalues
residus_multiple = modele_multiple.resid

fig, ax = plt.subplots(figsize=(9, 4))
ax.scatter(predictions_multiple, residus_multiple, s=10, alpha=0.15)
ax.axhline(0, color='red', linestyle='--')
ax.set(
    xlabel='Temps prédit (heures)',
    ylabel='Résidu (heures)',
    title='Résidus de la régression multiple'
)
plt.tight_layout()
plt.show()

# Examiner la forme de la distribution des résidus.
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(residus_multiple, bins=40, edgecolor='white')
ax.set(
    xlabel='Résidu (heures)',
    ylabel='Nombre d’athlètes',
    title='Distribution des résidus de la régression multiple'
)
plt.tight_layout()
plt.show()


#### <u>*Les temps moyens diffèrent-ils selon le pays ? — ANOVA de Welch*</u> ####

Dans la partie descriptive, nous avons comparé les **médianes des performances** par pays. Ici, nous cherchons à savoir si les **moyennes** sont différentes entre les pays assez représentés. Comme pour le test de genre, nous retenons **la meilleure performance enregistrée par athlète identifiable**, afin de limiter les observations répétées.

Pour ne pas multiplier les comparaisons sur de très petits groupes, nous conservons les pays avec **au moins 100 athlètes distincts**. Ce seuil est un choix méthodologique, et non un seuil statistique universel.

- **H₀ :** les temps moyens des pays sélectionnés sont tous égaux.
- **H₁ :** au moins un de ces pays présente un temps moyen différent.
- **Seuil de significativité :** α = 0,05.
- **Méthode :** ANOVA de Welch, extension de la comparaison de moyennes à plusieurs groupes, sans supposer l'égalité des variances. Les groupes doivent toutefois comporter des observations suffisamment indépendantes.

Il ne s'agit **pas** d'un test sur les médianes : nous comparons ici les moyennes. Une ANOVA significative ne dit pas à elle seule *quels* pays diffèrent. Les données ne sont pas un échantillon aléatoire représentatif des coureurs de chaque pays.

In [ ]:
from statsmodels.stats.oneway import anova_oneway

# Préparer une performance par athlète, en gardant son pays associé.
donnees_pays = dataFrame[[
    'Name', 'Date of Birth', 'Gender', 'Country', 'Age'
]].copy()
donnees_pays['Temps'] = temps

donnees_pays = donnees_pays.dropna()
donnees_pays = donnees_pays[
    donnees_pays['Gender'].isin(['Men', 'Women'])
]

donnees_pays = (
    donnees_pays.sort_values('Temps')
    .drop_duplicates(subset=['Name', 'Date of Birth', 'Gender'])
    .copy()
)

# Sélectionner les pays avec au moins 100 athlètes identifiables.
effectifs_athletes = donnees_pays['Country'].value_counts()
pays_analyse = effectifs_athletes[effectifs_athletes >= 100].index.tolist()

donnees_pays = donnees_pays[
    donnees_pays['Country'].isin(pays_analyse)
].copy()

# Résumer les temps par pays (en heures).
resume_inferentiel_pays = donnees_pays.groupby('Country')['Temps'].agg(
    effectif='count',
    moyenne='mean',
    mediane='median',
    ecart_type='std'
).sort_values('moyenne')

print('Pays retenus :', ', '.join(pays_analyse))
print('Athlètes retenus :', len(donnees_pays))
display(resume_inferentiel_pays.round(3))

# ANOVA de Welch : comparaison simultanée de plusieurs moyennes.
groupes_pays = [
    groupe['Temps'].to_numpy()
    for _, groupe in donnees_pays.groupby('Country')
]

resultat_anova = anova_oneway(groupes_pays, use_var='unequal')
f_observe = resultat_anova.statistic
p_anova = resultat_anova.pvalue
ddl_num, ddl_den = resultat_anova.df

alpha = 0.05
f_critique = stats.f.ppf(1 - alpha, ddl_num, ddl_den)

print(f'F observé : {f_observe:.3f}')
print(f'Degrés de liberté : {ddl_num:.0f} et {ddl_den:.1f}')
print(f'P-valeur : {p_anova:.3e}')
print(f'Valeur critique de F : {f_critique:.3f}')
print('Décision par p-valeur :',
      'rejet de H0' if p_anova < alpha else 'non-rejet de H0')
print('Décision par région de rejet (F > F critique) :',
      'rejet de H0' if f_observe > f_critique else 'non-rejet de H0')

**Interprétation de l'ANOVA de Welch :** le test compare les moyennes des six pays retenus, mais **sans tenir compte de la répartition par âge et par genre**. Même si H₀ est rejetée, on ne peut ni conclure que tous les pays sont différents, ni attribuer les écarts à la nationalité. La régression multiple suivante tient compte de l'âge et du genre pour affiner la comparaison.

**Résultat obtenu :** F ≈ 36,51, p ≈ 5,70 × 10⁻³², valeur critique F ≈ 2,232. Les deux méthodes de décision conduisent à **rejeter H₀** au seuil de 5 %. Il existe donc une différence entre au moins deux moyennes parmi les six pays analysés, sans pouvoir préciser les paires de pays par ce test seul.

#### <u>*Régression multiple : âge, genre et pays*</u> ####

Nous complétons le modèle de la section 4.4 avec la variable qualitative `Country`. Le modèle estime le temps de course en fonction de **l'âge**, du **genre** et du **pays**, simultanément :

\[
Temps = \beta_0 + \beta_1 \times Âge + \beta_2 \times Genre + \sum_j \gamma_j Pays_j + \varepsilon
\]

Nous conservons exactement les **mêmes athlètes et les mêmes six pays** que pour l'ANOVA afin de rendre les analyses comparables.

- **Références :** `Men` pour le genre et `KEN` (Kenya) pour le pays. Les autres coefficients indiquent des différences de temps **par rapport au Kenya**, à âge et genre identiques.
- **Pour le facteur pays :** H₀ : tous les coefficients de pays sont nuls ; H₁ : au moins un coefficient n'est pas nul.
- **Test global du facteur pays :** test de Wald sur les coefficients de pays, avec erreurs standards robustes HC3.
- **Comparaison de $R^2$ :** modèle âge + genre, puis modèle âge + genre + pays, tous deux sur le même sous-ensemble.

Ces coefficients décrivent des **associations ajustées**. Ils ne prouvent pas une influence causale de la nationalité ; l'année et la course, l'entraînement et la manière dont les performances ont été sélectionnées peuvent contribuer aux différences observées.

In [ ]:
# Ajuster les deux modèles sur les mêmes athlètes.
modele_sans_pays = smf.ols(
    'Temps ~ Age + C(Gender, Treatment(reference="Men"))',
    data=donnees_pays
).fit(cov_type='HC3')

modele_avec_pays = smf.ols(
    'Temps ~ Age + C(Gender, Treatment(reference="Men")) '
    '+ C(Country, Treatment(reference="KEN"))',
    data=donnees_pays
).fit(cov_type='HC3')

# Présenter les coefficients et intervalles de confiance en minutes.
ic_coefficients = modele_avec_pays.conf_int()
resume_coefficients = pd.DataFrame({
    'Coefficient (min)': modele_avec_pays.params * 60,
    'IC 95 % - bas (min)': ic_coefficients[0] * 60,
    'IC 95 % - haut (min)': ic_coefficients[1] * 60,
    'P-valeur': modele_avec_pays.pvalues
})

print(f'Effectif : {len(donnees_pays)} athlètes')
print(f'R² âge + genre : {modele_sans_pays.rsquared:.4f}')
print(f'R² âge + genre + pays : {modele_avec_pays.rsquared:.4f}')
print(f'R² ajusté du modèle complet : {modele_avec_pays.rsquared_adj:.4f}')
print('\nCoefficients du modèle complet :')
display(resume_coefficients.round(4))

# Vérifier globalement si la variable pays apporte de l'information.
tests_termes = modele_avec_pays.wald_test_terms(scalar=True).table
nom_terme_pays = next(
    nom for nom in tests_termes.index if 'Country' in nom
)

statistique_pays = tests_termes.loc[nom_terme_pays, 'statistic']
p_valeur_pays = tests_termes.loc[nom_terme_pays, 'pvalue']

print(f'\nTest global du facteur pays (Wald χ²) : {statistique_pays:.3f}')
print(f'P-valeur du facteur pays : {p_valeur_pays:.3e}')
print('Décision au seuil de 5 % :',
      'rejet de H0' if p_valeur_pays < 0.05 else 'non-rejet de H0')

#### <u>*Visualisation des écarts de temps estimés entre pays*</u> ####

Le graphique ci-dessous montre les **coefficients ajustés** associés aux pays, en **minutes par rapport au Kenya**, avec leurs intervalles de confiance à 95 %. Le Kenya sert uniquement de **référence mathématique**, pas de classement absolu des pays.

Un coefficient positif signifie un temps prédit plus long que pour le Kenya à âge et genre identiques ; un coefficient négatif signifie un temps prédit plus court. Les intervalles permettent de visualiser l'incertitude des estimations.

In [ ]:
# Tracer seulement les coefficients des pays (pas l'interception ni l'âge).
coefficients_pays = resume_coefficients.loc[
    resume_coefficients.index.str.contains('Country')
].copy()

# Extraire les codes pays des noms de coefficients.
coefficients_pays.index = [
    nom.split('[T.')[-1].rstrip(']')
    for nom in coefficients_pays.index
]
coefficients_pays = coefficients_pays.sort_values('Coefficient (min)')

estimations = coefficients_pays['Coefficient (min)']
erreur_gauche = estimations - coefficients_pays['IC 95 % - bas (min)']
erreur_droite = coefficients_pays['IC 95 % - haut (min)'] - estimations

fig, ax = plt.subplots(figsize=(9, 4))
ax.errorbar(
    estimations,
    coefficients_pays.index,
    xerr=[erreur_gauche, erreur_droite],
    fmt='o',
    capsize=4
)
ax.axvline(0, color='gray', linestyle='--', linewidth=1)
ax.set(
    xlabel='Différence estimée par rapport au Kenya (minutes)',
    ylabel='Pays',
    title='Écarts ajustés par pays (âge et genre pris en compte)'
)
ax.grid(axis='x', alpha=0.2)
plt.tight_layout()
plt.show()

#### <u>*Interprétation et limites de l'analyse par pays*</u> ####

- **ANOVA de Welch :** elle vérifie si toutes les moyennes par pays peuvent être considérées égales, mais ne contrôle pas l'âge ni le genre.
- **Résultat du modèle complet :** $R^2$ ≈ 0,8979 contre $R^2$ ≈ 0,8882 pour le modèle âge + genre **sur les mêmes 3 089 athlètes**. Le gain de $R^2$ est donc d’environ 0,0097 (0,97 point de pourcentage). Le test global de Wald donne χ² ≈ 304,23 et p ≈ 1,23 × 10⁻⁶³ ; les coefficients des pays ne sont pas tous nuls selon le modèle. Un effet statistique global ne signifie pas nécessairement une grande différence pratique.
- **Régression multiple :** elle compare les pays en tenant compte de l'âge et du genre. Le test global du facteur `Country` sert à déterminer si les coefficients de pays sont conjointement nuls.
- **Lecture des coefficients :** ils représentent des écarts moyens estimés **en minutes**, relativement au Kenya, à âge et genre identiques ; ils ne correspondent pas aux écarts entre médianes de la partie descriptive.
- **Prudence :** ces données sont composées de performances sélectionnées. Les différences entre pays peuvent refléter les courses, les années, la structure des athlètes et d'autres facteurs non mesurés. Même avec un petit p, un effet observé ne démontre aucune causalité.

**Décision de gestion :** ne pas fixer de critères de performance sur la seule base de la nationalité. Pour analyser des temps de référence, il faudrait au minimum distinguer les catégories sportives pertinentes et, si possible, les années et les courses.

## <u>5. Interprétation et conclusion</u> ##

#### <u>*Synthèse des résultats*</u> ####

Notre étude avait pour objectif de déterminer dans quelle mesure l'âge, le genre et le pays sont associés aux performances des marathoniens, mesurées par leur temps de course.

**Concernant le genre**, nous avons observé une différence importante entre les performances masculines et féminines. Le test de Welch a confirmé que leurs temps moyens sont statistiquement différents, avec un écart d'environ 16,5 minutes en faveur des hommes dans notre échantillon.

**Concernant l'âge**, le coefficient de corrélation de Pearson est extrêmement faible (r = 0,023). La régression linéaire simple montre également que l'âge explique une très faible proportion de la variation des temps ($R^2$ ≈ 0,0005). Lorsque le genre est pris en compte dans la régression multiple, l'association linéaire avec l'âge n'est plus statistiquement significative au seuil de 5 % (p = 0,0755).

**Concernant le pays**, l'ANOVA de Welch indique une différence statistiquement significative entre les temps moyens d'au moins deux des six pays sélectionnés. La régression multiple montre que l'ajout du pays améliore légèrement le modèle, avec une augmentation du $R^2$ de 88,82 % à 89,79 %, soit environ 0,97 point de pourcentage. Cette comparaison a été réalisée sur les mêmes 3 089 athlètes.

#### <u>*Réponse à la problématique*</u> ####

Les résultats montrent que le genre est la variable la plus fortement associée aux différences de temps de course dans notre jeu de données.

Le pays présente également une association statistique avec les performances, mais son apport supplémentaire au modèle reste relativement limité lorsque l'âge et le genre sont pris en compte.

En revanche, l'âge seul ne semble pas être un bon prédicteur linéaire des performances.

Ces résultats mettent en évidence des associations entre certaines caractéristiques et les temps de course, mais ne permettent pas d'établir des relations de cause à effet.

#### <u>*Limites de notre étude*</u> ####

Notre analyse présente plusieurs limites.

Tout d'abord, le jeu de données contient principalement des performances de marathoniens de haut niveau. Les résultats ne sont donc pas représentatifs de l'ensemble des participants aux marathons.

Ensuite, les effectifs sont inégalement répartis selon le genre, l'âge et le pays. Certains groupes sont beaucoup moins représentés, ce qui rend les comparaisons moins fiables.

De plus, plusieurs performances peuvent appartenir à un même athlète. Pour certaines analyses inférentielles, nous avons conservé uniquement la meilleure performance de chaque coureur identifiable afin de limiter ce problème. Cependant, l'identification des athlètes n'est pas toujours parfaitement fiable et le meilleur résultat enregistré peut dépendre du nombre de courses présentes dans le fichier.

Enfin, les données proviennent de différentes années et compétitions. Les caractéristiques des parcours, les conditions météorologiques, l'expérience et le niveau d'entraînement peuvent influencer les performances, mais ne sont pas directement pris en compte dans nos modèles.

Ces limites nous obligent à interpréter les résultats avec prudence et à éviter toute généralisation à l'ensemble des marathoniens.

#### <u>*Conclusion générale et décisions*</u> ####

Cette étude nous a permis d'appliquer différents outils de statistiques descriptives et inférentielles afin de mieux comprendre les facteurs associés aux performances sur marathon.

Nous avons constaté que le genre est fortement associé aux différences de temps dans notre échantillon, tandis que l'âge présente une relation linéaire très faible avec les performances. Des différences statistiquement significatives existent également entre certains pays, mais leur interprétation nécessite de tenir compte des autres caractéristiques des athlètes.

**Sur le plan de la gestion sportive**, nous recommandons de comparer les performances à l'aide de catégories adaptées, notamment selon le genre et, lorsque cela est pertinent, la tranche d'âge. Les effectifs et les critères de sélection des performances devraient également être précisés.

Nous déconseillons d'utiliser uniquement l'âge ou la nationalité pour évaluer le potentiel d'un marathonien ou fixer des critères de sélection.

Pour améliorer la fiabilité de futures analyses, il serait intéressant de disposer d'un jeu de données plus représentatif, comprenant également l'entraînement, l'expérience, les caractéristiques du parcours et les conditions météorologiques.

Ainsi, notre travail montre l'intérêt des outils statistiques pour analyser les performances sportives, tout en soulignant l'importance de tenir compte des limites des données avant de prendre des décisions.

### Rappel pour la lecture des résultats

- $\bar{x}$ : moyenne de l'échantillon ; $s$ : écart-type de l'échantillon ; $n$ : effectif.
- $H_0$ : hypothèse nulle ; $H_1$ : hypothèse alternative ; $\alpha$ : seuil de significativité ; $p$ : p-valeur.
- $r$ : corrélation de Pearson ; $\rho$ : corrélation de Spearman.
- $R^2$ : coefficient de détermination ; $\beta_j$ : coefficients du modèle de régression.
- $t$ : statistique d'un test t ; $F$ : statistique d'une ANOVA ; $\chi^2$ : statistique du test de Wald.

**Important :** après toute modification du fichier CSV ou du code, exécutez toutes les cellules dans l'ordre. Les valeurs écrites manuellement dans les paragraphes d'interprétation doivent toujours être comparées aux sorties recalculées.